# Research Questions Analysis — Restaurant Ratings (Milan)

**Pipeline Stage 7 — Analysis & results.** Read-only analytics over the flat
ClickHouse tables produced by `dataman-load-clickhouse`.

The analysis is split **one notebook per research question** — `q01_*` … `q11_*` (plus this
`q00_overview`) — each self-contained: it opens with `from analysis.notebook import *` (shared
ClickHouse client, `run`/`publish` helpers, palettes and logo helpers), then the SQL, the
rendered result, charts, and a written answer. The questions are README's **4 main + 3
secondary** plus **4 extended, feature-driven** ones. Result tables are exported to
`report/final/tables/` as **CSV + LaTeX** for the final report. This overview notebook holds
the methodology notes and the platform-coverage baseline (Q0).

### Prerequisites
1. Start the analytics layer: `docker compose --profile analytics up -d clickhouse`
2. Load the data: `uv run dataman-load-clickhouse all`
   *(the notebooks rely on the extended `restaurants_integrated` schema — per-platform
   photo counts, contact-completeness flags, cuisines and `price_tier`; if you changed the
   loader, drop & reload: `DROP TABLE restaurants_integrated` then reload).*
3. Install notebook deps: `uv sync --extra analysis`
4. Run a notebook top to bottom, or execute one (e.g.
   `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/q01_consistency.ipynb`),
   or all at once: `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/q*.ipynb`.

> **Read-only.** Every query is a `SELECT`; the notebooks perform no writes, DDL or
> truncation against ClickHouse or MongoDB.

In [1]:
from analysis.notebook import *

Connected to ClickHouse db='dataman' at localhost:8123


## Methodology & data structure

Two facts shape every cross-platform comparison below:

1. **Google is the seed.** Every integrated restaurant has a Google record; Tripadvisor
   and TheFork are matched onto it. So the meaningful comparisons are the *pairs*
   **Google–Tripadvisor**, **Google–TheFork**, and (only within the all-three set)
   **Tripadvisor–TheFork**. We report consistency **pairwise** rather than as one blob,
   because TheFork's coverage (~0.9k matched venues) is an order of magnitude smaller than
   Tripadvisor's (~3.9k).
2. **"Total reviews" is not a clean popularity measure.** Summing review counts across
   platforms mixes three different, overlapping user bases and silently treats a missing
   platform as zero. Where a single popularity proxy is needed we use the **Google review
   count** (widest, most complete coverage) and say so; combined sums are avoided.

Consistency metrics use only venues rated on **≥ 2 platforms** (`rating_platform_count >= 2`).

In [2]:
coverage = run(queries.q0_platform_coverage())
publish(coverage, "q0_platform_coverage", "Platform co-occurrence across integrated restaurants.")
coverage

,has_google,has_tripadvisor,has_thefork,restaurants
0,1,0,0,5967
1,1,1,0,3179
2,1,1,1,745
3,1,0,1,163
